# 25 - Tuning with the team-strength features (experiment 12)

The parameters of submissions 22-24 were tuned in experiment 04, **before** the team-strength features existed
(experiment 11). The new features are among the strongest, so the best parameters may have moved. This notebook
re-tunes only the models of those submissions: **LightGBM, XGBoost and CatBoost** (scikit-learn's Gradient Boosting
is slow and weighs a quarter of the vote: it keeps its parameters).

Protocol, to avoid the optimistic estimates of experiment 04 (XGBoost: 0.6828 in the search, 0.6751 in nested CV):
1. **Search** (`RandomizedSearchCV`) on the 5 folds of **seed 42** only, in a narrow space around the current values.
   The team-strength encoder is the first step of the pipeline, so it is refitted in every fold; for LightGBM its
   parameters (smoothing K, share of training rows marked unseen) are searched too.
2. **Check** on the folds of **seeds 7 and 123** (never used by the search): tuned vs current parameters, paired.
3. **Adopt** the tuned parameters of a model only if they gain **≥ +0.3 points** on those 10 folds and improve
   **at least 7 of 10**; for LightGBM also: with 100% unseen teams not worse than the current configuration.
4. The soft vote with the adopted parameters is compared with the current vote on the same 10 folds.

Features: v1 (155) + team keys, as submissions 22 and 24.

## 0. Setup

In [1]:
import os
import sys
import json
import time
import warnings
from pathlib import Path

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features import team_key_features
from src.models import get_model_specs, params_to_json, params_to_text, with_params
from src.team_strength import KEY_P1, KEY_P2, mask_teams, with_team_strength
from src.validation import SEED, augmented_cv_scores, cv_scores

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
PROC = ROOT / "data" / "processed"
OUT = PROC / "models"
OUT.mkdir(parents=True, exist_ok=True)

team, states = [pd.read_pickle(PROC / f"{name}_train.pkl") for name in ("team", "states")]
ref = pd.read_pickle(PROC / "features_train.pkl")             # v1 features (notebook 03)
with open(PROC / "feature_groups.json") as f:
    groups = json.load(f)
y = ref.pop("player_won")
NUM = [c for c in ref.columns if c not in groups["species_cat"]]
KEYS = [KEY_P1, KEY_P2]
X = pd.concat([ref[NUM], team_key_features(team, states).reindex(ref.index)], axis=1)
assert X[KEYS].notna().all().all()
print(f"battles: {len(X):,} | features: {len(NUM)} + team keys")

# Current parameters (experiment 04) and encoder (notebook 21, section 6), as in submissions 22-24
PARAMS = {
    "xgboost": {"model__colsample_bytree": 0.6856273981493477, "model__gamma": 0.010493314645539311,
                "model__learning_rate": 0.012145928554953287, "model__max_depth": 6,
                "model__min_child_weight": 1.3386261584543908, "model__n_estimators": 720,
                "model__reg_lambda": 3.5416296712651336, "model__subsample": 0.5993578407670862},
    "lightgbm": {"model__colsample_bytree": 0.6286197472598067, "model__learning_rate": 0.006552992991573526,
                 "model__min_child_samples": 106, "model__n_estimators": 1424, "model__num_leaves": 12,
                 "model__reg_alpha": 0.6070874110546393, "model__reg_lambda": 2.7129089558470034,
                 "model__subsample": 0.6629794526009424},
    "catboost": {"model__bagging_temperature": 0.15071754396542947, "model__depth": 5, "model__iterations": 763,
                 "model__l2_leaf_reg": 1.14876138417489, "model__learning_rate": 0.03898458749305194,
                 "model__random_strength": 1.3551287236845648},
    "gradient_boosting": {"model__learning_rate": 0.04760153910846178, "model__max_depth": 3,
                          "model__max_features": None, "model__min_samples_leaf": 91, "model__n_estimators": 466,
                          "model__subsample": 0.9314950036607718},
}
ENCODER = {"train_unseen": 0.3}
specs = get_model_specs()

def pipeline(name, params, encoder=ENCODER):
    return with_team_strength(with_params(specs[name], params), **encoder)

CHECK_SEEDS = (7, 123)

def check_scores(model, val_transform=None):
    # 10 folds never used by the search (seeds 7 and 123); same folds with or without val_transform
    if val_transform is None:
        return np.concatenate([cv_scores(model, X, y, seed=s) for s in CHECK_SEEDS])
    return np.concatenate([augmented_cv_scores(model, X, y, seed=s, val_transform=val_transform) for s in CHECK_SEEDS])

battles: 10,000 | features: 155 + team keys


## 1. Search on the seed-42 folds

Narrow spaces around the current values (the search of experiment 04 put LightGBM at the edge of its space: lowest
learning rate, most trees, so the space is extended in that direction).

In [2]:
SPACES = {
    "lightgbm": {
        "model__n_estimators": randint(800, 3000), "model__learning_rate": loguniform(2e-3, 2e-2),
        "model__num_leaves": randint(6, 32), "model__min_child_samples": randint(40, 220),
        "model__subsample": uniform(0.5, 0.4), "model__colsample_bytree": uniform(0.3, 0.6),
        "model__reg_alpha": loguniform(1e-2, 5), "model__reg_lambda": loguniform(1e-1, 20),
        "team__smoothing": [5.0, 10.0, 20.0, 30.0], "team__train_unseen": [0.2, 0.3, 0.4],
    },
    "xgboost": {
        "model__n_estimators": randint(400, 1800), "model__learning_rate": loguniform(4e-3, 3e-2),
        "model__max_depth": randint(3, 8), "model__min_child_weight": loguniform(1, 30),
        "model__subsample": uniform(0.5, 0.4), "model__colsample_bytree": uniform(0.4, 0.5),
        "model__gamma": loguniform(1e-3, 1), "model__reg_lambda": loguniform(0.5, 20),
    },
    "catboost": {
        "model__iterations": randint(500, 1600), "model__learning_rate": loguniform(1.5e-2, 8e-2),
        "model__depth": randint(4, 8), "model__l2_leaf_reg": loguniform(1, 20),
        "model__random_strength": uniform(0, 2), "model__bagging_temperature": uniform(0, 1),
    },
}
N_ITER = {"lightgbm": 40, "xgboost": 30, "catboost": 20}

cv42 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
searches = {}
for name in SPACES:
    t0 = time.time()
    search = RandomizedSearchCV(pipeline(name, PARAMS[name]), SPACES[name], n_iter=N_ITER[name], scoring="accuracy",
                                cv=cv42, n_jobs=-1, refit=False, random_state=SEED, error_score=np.nan)
    search.fit(X, y)
    searches[name] = search
    res = search.cv_results_
    i = int(np.nanargmax(res["mean_test_score"]))
    print(f"{name:9s} best on seed 42: {res['mean_test_score'][i]:.4f} ± {res['std_test_score'][i]:.4f} "
          f"({time.time() - t0:.0f}s) | {params_to_text(res['params'][i])}")

lightgbm  best on seed 42: 0.6903 ± 0.0052 (109s) | colsample_bytree=0.6432, learning_rate=0.01174, min_child_samples=56, n_estimators=2043, num_leaves=19, reg_alpha=2.946, reg_lambda=0.7108, subsample=0.5062, team__smoothing=30, team__train_unseen=0.2
xgboost   best on seed 42: 0.6908 ± 0.0039 (73s) | colsample_bytree=0.7609, gamma=0.005105, learning_rate=0.006701, max_depth=5, min_child_weight=5.918, n_estimators=1438, reg_lambda=0.7527, subsample=0.6757
catboost  best on seed 42: 0.6914 ± 0.0032 (179s) | bagging_temperature=0.05808, depth=7, iterations=1371, l2_leaf_reg=8.341, learning_rate=0.01553, random_strength=1.94


In [3]:
# Current parameters on the same seed-42 folds, for reference
for name in SPACES:
    s = cv_scores(pipeline(name, PARAMS[name]), X, y)
    print(f"{name:9s} current on seed 42: {s.mean():.4f} ± {s.std():.4f}")

lightgbm  current on seed 42: 0.6905 ± 0.0045
xgboost   current on seed 42: 0.6883 ± 0.0045
catboost  current on seed 42: 0.6849 ± 0.0102


## 2. Check on the folds of seeds 7 and 123

The 3 best candidates of each search (not only the first: the ranking on 5 folds is noisy) are compared with the
current parameters on 10 folds the search never saw.

In [4]:
rows, best_params = [], {}
for name, search in searches.items():
    t0 = time.time()
    res = pd.DataFrame(search.cv_results_).sort_values("mean_test_score", ascending=False).head(3)
    current = check_scores(pipeline(name, PARAMS[name]))
    candidates = []
    for rank, (_, r) in enumerate(res.iterrows(), start=1):
        p = r["params"]
        enc = {k.replace("team__", ""): v for k, v in p.items() if k.startswith("team__")}
        model_p = {k: v for k, v in p.items() if k.startswith("model__")}
        tuned = check_scores(pipeline(name, {**PARAMS[name], **model_p}, {**ENCODER, **enc}))
        d = tuned - current
        row = {"model": name, "candidate": rank, "seed 42 (search)": r["mean_test_score"], "current (7, 123)": current.mean(),
               "tuned (7, 123)": tuned.mean(), "gain (points)": d.mean() * 100, "folds improved": int((d > 0).sum()),
               "params": params_to_text(p)}
        rows.append(row)
        candidates.append((d.mean(), int((d > 0).sum()), model_p, enc))
    gain, wins, model_p, enc = max(candidates, key=lambda c: c[0])
    best_params[name] = {"gain": gain * 100, "wins": wins, "model": model_p, "encoder": enc}
    print(f"{name:9s} checked in {time.time() - t0:.0f}s")
check = pd.DataFrame(rows)
check.round(4)

lightgbm  checked in 38s
xgboost   checked in 51s
catboost  checked in 274s


,model,candidate,seed 42 (search),"current (7, 123)","tuned (7, 123)",gain (points),folds improved,params
0,lightgbm,1,0.6903,0.6848,0.6792,-0.550,1,"colsample_bytree=0.6432, learning_rate=0.01174..."
1,lightgbm,2,0.6891,0.6848,0.6823,-0.245,3,"colsample_bytree=0.7775, learning_rate=0.01553..."
2,lightgbm,3,0.6886,0.6848,0.6868,0.210,6,"colsample_bytree=0.6256, learning_rate=0.00386..."
3,xgboost,1,0.6908,0.6856,0.6859,0.035,5,"colsample_bytree=0.7609, gamma=0.005105, learn..."
4,xgboost,2,0.6908,0.6856,0.6844,-0.115,4,"colsample_bytree=0.4527, gamma=0.02342, learni..."
5,xgboost,3,0.6907,0.6856,0.6878,0.215,5,"colsample_bytree=0.6648, gamma=0.005316, learn..."
6,catboost,1,0.6914,0.6873,0.6892,0.195,6,"bagging_temperature=0.05808, depth=7, iteratio..."
7,catboost,2,0.6908,0.6873,0.6878,0.055,3,"bagging_temperature=0.8631, depth=7, iteration..."
8,catboost,3,0.6895,0.6873,0.6856,-0.175,4,"bagging_temperature=0.6833, depth=7, iteration..."


## 3. Decision per model, and teams never seen (LightGBM)

Adopt if gain ≥ +0.3 points on the 10 check folds with ≥ 7/10 improved. For LightGBM, whose encoder parameters were
searched too, also: with 100% unseen teams not worse than the current configuration.

In [5]:
adopted = {}
for name, b in best_params.items():
    ok = b["gain"] >= 0.3 and b["wins"] >= 7
    if ok and name == "lightgbm" and b["encoder"]:
        unseen = lambda v: mask_teams(v, 1.0, SEED)
        cur_u = check_scores(pipeline(name, PARAMS[name]), unseen).mean()
        new_u = check_scores(pipeline(name, {**PARAMS[name], **b["model"]}, {**ENCODER, **b["encoder"]}), unseen).mean()
        print(f"lightgbm with 100% unseen teams: current {cur_u:.4f} | tuned {new_u:.4f}")
        ok = new_u >= cur_u - 0.001
    print(f"{'ADOPT' if ok else 'keep '} {name:9s} gain {b['gain']:+.2f} points, {b['wins']}/10 folds")
    if ok:
        adopted[name] = b

keep  lightgbm  gain +0.21 points, 6/10 folds
keep  xgboost   gain +0.21 points, 5/10 folds
keep  catboost  gain +0.19 points, 6/10 folds


## 4. Soft vote: adopted parameters vs current

The vote of submission 22 (XGBoost, LightGBM, CatBoost, Gradient Boosting) with the adopted parameters, against the
current vote, on the same 10 check folds. The encoder of the vote is the LightGBM one when LightGBM was adopted.

In [6]:
def vote(params, encoder):
    members = [(n, with_params(specs[n], params[n])) for n in PARAMS]
    return with_team_strength(VotingClassifier(members, voting="soft", n_jobs=1), **encoder)

new_params = {n: {**PARAMS[n], **(adopted[n]["model"] if n in adopted else {})} for n in PARAMS}
new_encoder = {**ENCODER, **(adopted["lightgbm"]["encoder"] if "lightgbm" in adopted else {})}
if adopted:
    t0 = time.time()
    cur_v = check_scores(vote(PARAMS, ENCODER))
    new_v = check_scores(vote(new_params, new_encoder))
    d = new_v - cur_v
    print(f"soft vote: current {cur_v.mean():.4f} | tuned {new_v.mean():.4f} | gain {d.mean() * 100:+.2f} points, "
          f"{(d > 0).sum()}/10 folds ({time.time() - t0:.0f}s)")
else:
    print("no model adopted: the submissions keep the current parameters")

no model adopted: the submissions keep the current parameters


## 5. Parameters for the submission notebooks

In [7]:
final = {"params": {n: params_to_json(p) for n, p in new_params.items()}, "encoder": params_to_json(new_encoder),
         "adopted": sorted(adopted)}
with open(OUT / "best_params_ts.json", "w") as f:
    json.dump(final, f, indent=2)
check.to_csv(OUT / "tuning_ts_check.csv", index=False)
print(json.dumps(final, indent=2))

{
  "params": {
    "xgboost": {
      "model__colsample_bytree": 0.6856273981493477,
      "model__gamma": 0.010493314645539311,
      "model__learning_rate": 0.012145928554953287,
      "model__max_depth": 6,
      "model__min_child_weight": 1.3386261584543908,
      "model__n_estimators": 720,
      "model__reg_lambda": 3.5416296712651336,
      "model__subsample": 0.5993578407670862
    },
    "lightgbm": {
      "model__colsample_bytree": 0.6286197472598067,
      "model__learning_rate": 0.006552992991573526,
      "model__min_child_samples": 106,
      "model__n_estimators": 1424,
      "model__num_leaves": 12,
      "model__reg_alpha": 0.6070874110546393,
      "model__reg_lambda": 2.7129089558470034,
      "model__subsample": 0.6629794526009424
    },
    "catboost": {
      "model__bagging_temperature": 0.15071754396542947,
      "model__depth": 5,
      "model__iterations": 763,
      "model__l2_leaf_reg": 1.14876138417489,
      "model__learning_rate": 0.03898458749305194,
 

## 6. Summary

- **Search (seed 42):** the best candidates look better than the current parameters on the folds they were chosen
  on: LightGBM 0.6903 (current 0.6905), XGBoost 0.6908 (0.6883), CatBoost 0.6914 (0.6849).
- **Check (seeds 7 and 123, 10 folds never seen by the search):** none of them holds. Best of the 3 candidates per
  model: LightGBM +0.21 points (6/10 folds), XGBoost +0.21 (5/10), CatBoost +0.19 (6/10). The first LightGBM
  candidate, the best on seed 42, is the worst out of sample (-0.55, 1/10): the classic optimism of tuning on the same
  folds, as in experiment 04.
- **Decision:** no model reaches the rule (≥ +0.3 points with ≥ 7/10 folds): the submissions keep the parameters of
  experiment 04 and the encoder `train_unseen=0.3`. The current parameters were already near the optimum: the new
  features did not move it.
- On the check folds the single models with team strength score LightGBM 0.6848, XGBoost 0.6856, CatBoost 0.6873:
  within noise of each other.